# CHENTO_LIMIT_BID v3 — multi-asset backtest (BTC + ETH + OP)

v2 backtest on BTC alone showed +0.30R per trade × 5 trades/year = +3% annual.
The [comparison notebook](chento_vs_bot_comparison.ipynb) showed the gap to
chento's returns is dominated by frequency. v3 adds ETH and OP variants to
the same sleeve to see if multi-asset breadth closes the gap.

## Data caveats per asset

| Asset | Spot 15m | Perp 15m | OI | Funding | Basis | Spot CVD |
|---|---|---|---|---|---|---|
| BTC | ✓ `cd_spot_15m` | ✓ `cd_futures_15m` | ✓ `cd_open_interest` | ✓ `cd_funding_rate` | ✓ | ✓ |
| ETH | resampled from `eth_1m` | ✗ | ✗ | ✓ `cd_funding_rate_eth` | ✗ | ✗ |
| OP | ✗ | resampled from `op_perp_1m` | ✗ | ✗ | ✗ | ✗ |

So **BTC uses full v2 confluence (4 legs)**, **ETH uses funding-only (1 leg)**,
and **OP uses no confluence — pure MTF + base detection**. We expect per-trade
R to drop on ETH/OP but signal frequency to rise. Verdict at the end depends
on annual return (R × frequency).

v3.5 (next session, if v3 is promising) backfills the missing data.

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('dark_background')

ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent: raise RuntimeError('locate prod.db')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DB = ROOT / 'data' / 'databases' / 'prod.db'

from strategies.sleeves.chento_limit_bid import math as cli_math
from strategies.sleeves.chento_limit_bid import config as cli_cfg
print(f'DB: {DB}')

## Asset-specific data loaders

In [ ]:
def _load_table(table, ts_col='timestamp', ts_unit='s'):
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(f'SELECT * FROM {table} ORDER BY {ts_col}', con)
    con.close()
    df['ts'] = pd.to_datetime(df[ts_col], unit=ts_unit, utc=True)
    return df.set_index('ts')[lambda d: ~d.index.duplicated(keep='last')]

def _load_1m(table):
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(
        f'SELECT open_time, open, high, low, close, volume FROM {table} ORDER BY open_time',
        con)
    con.close()
    df['ts'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    df = df.set_index('ts').drop(columns='open_time')
    df.columns = ['o','h','l','c','v']
    return df[~df.index.duplicated(keep='last')]

def _resample(df, rule):
    return df.resample(rule).agg(o=('o','first'), h=('h','max'),
                                  l=('l','min'),   c=('c','last'),
                                  v=('v','sum')).dropna()

def build_btc_frame():
    spot15 = _load_table('cd_spot_15m')
    fut15  = _load_table('cd_futures_15m')
    oi_h   = _load_table('cd_open_interest')
    fund_h = _load_table('cd_funding_rate')
    start = oi_h.index.min()
    spot15 = spot15.loc[start:]; fut15 = fut15.loc[start:]; fund_h = fund_h.loc[start:]
    f = pd.DataFrame(index=spot15.index)
    f['spot_o']=spot15['open']; f['spot_h']=spot15['high']
    f['spot_l']=spot15['low'];  f['spot_c']=spot15['close']
    f['spot_cvd']=spot15['volume_buy']-spot15['volume_sell']
    f['fut_c']=fut15['close'].reindex(f.index)
    f['basis_bp']=(f['fut_c']-f['spot_c'])/f['spot_c']*10000.0
    f['oi']=oi_h['oi_close'].reindex(f.index).ffill(limit=4)
    f['funding']=fund_h['fr_close'].reindex(f.index).ffill(limit=32)
    return f.dropna(subset=['spot_c','fut_c','oi','funding']).copy()

def build_eth_frame():
    # Resample eth_1m → 15m for spot OHLC. Funding from cd_funding_rate_eth.
    # No basis (no ETH perp 15m), no OI, no spot CVD.
    m1 = _load_1m('eth_1m')
    f15 = _resample(m1, '15min')
    fund_h = _load_table('cd_funding_rate_eth')
    f = pd.DataFrame(index=f15.index)
    f['spot_o']=f15['o']; f['spot_h']=f15['h']
    f['spot_l']=f15['l']; f['spot_c']=f15['c']
    f['spot_cvd'] = 0.0      # no CVD available — score leg fails (0)
    f['fut_c'] = f['spot_c'] # no perp — treat basis as 0
    f['basis_bp'] = 0.0
    f['oi'] = 1.0            # constant — OI flush leg fails (0)
    f['funding'] = fund_h['fr_close'].reindex(f.index).ffill(limit=32).fillna(0.0)
    return f.dropna(subset=['spot_c']).copy()

def build_op_frame():
    # Resample op_perp_1m → 15m. No funding, no OI, no basis, no CVD.
    m1 = _load_1m('op_perp_1m')
    f15 = _resample(m1, '15min')
    f = pd.DataFrame(index=f15.index)
    f['spot_o']=f15['o']; f['spot_h']=f15['h']
    f['spot_l']=f15['l']; f['spot_c']=f15['c']
    f['spot_cvd']=0.0; f['fut_c']=f['spot_c']
    f['basis_bp']=0.0; f['oi']=1.0; f['funding']=0.0
    return f.dropna(subset=['spot_c']).copy()

print('Loading per-asset 15m frames...')
asset_frames = {}
for name, loader in [('BTC', build_btc_frame),
                       ('ETH', build_eth_frame),
                       ('OP',  build_op_frame)]:
    f = loader()
    asset_frames[name] = f
    print(f'  {name}: {len(f):,} bars, {f.index.min()} → {f.index.max()}')

## MTF bias per asset

Resample each asset's 1m OHLCV to M/W/D/4H/1H, compute bias series.

In [ ]:
def build_mtf_bias(asset_1m_table):
    m1 = _load_1m(asset_1m_table)
    rules = {'M':'1ME','W':'1W','D':'1D','H4':'4h','H1':'1h'}
    bias_map = {}
    for label, rule in rules.items():
        cfg = cli_cfg.MTF_DEFS[label]
        tf_df = _resample(m1, rule)
        bias_map[label] = cli_math.compute_tf_bias_series(
            tf_df, period=cfg['period'], slope=cfg['slope'])
    return bias_map

asset_mtf_maps = {
    'BTC': build_mtf_bias('btc_1m'),
    'ETH': build_mtf_bias('eth_1m'),
    'OP':  build_mtf_bias('op_perp_1m'),
}
for asset, bm in asset_mtf_maps.items():
    sizes = ', '.join(f"{k}={len(s)}" for k, s in bm.items())
    print(f'{asset}: {sizes}')

## Per-asset backtest function

Same v2 logic, parameterized by asset. For ETH/OP we DROP the confluence
gate since we don't have basis/OI/CVD.

In [ ]:
def run_backtest(asset_name, f, mtf_bias_map, use_confluence=True):
    signals = []
    last_trigger = None
    cooldown_td = pd.Timedelta(minutes=cli_cfg.COOLDOWN_MIN)
    f_idx = f.index
    cost_per_unit = (cli_cfg.COST_BP_RT + cli_cfg.SLIPPAGE_BP_RT) / 10000.0
    min_bars = cli_cfg.BASE_WINDOW_HOURS * 4 + cli_cfg.BASE_EXPANSION_DAYS * 24 * 4 + 12
    for i in range(len(f_idx)):
        now_ts = f_idx[i]
        now_dt = now_ts.to_pydatetime()
        if last_trigger is not None and (now_ts - last_trigger) < cooldown_td: continue
        if not cli_math.passes_time_gate(now_dt): continue
        if i < min_bars: continue
        sub = f.iloc[:i+1]
        base = cli_math.detect_active_base(sub, now_ts)
        if base is None: continue
        current_price = float(sub['spot_c'].iloc[-1])
        if not cli_math.is_approaching_base(current_price, base['base_low'],
                                              cli_cfg.BASE_APPROACH_BAND_PCT): continue
        if use_confluence:
            window = f.loc[base['base_start_ts']:base['base_end_ts']]
            score = cli_math.score_base_window(window)
            if score['conf_score'] < cli_cfg.CONF_SCORE_MIN: continue
            cf = score['conf_score']
        else:
            cf = None
        sig, net = cli_math.mtf_signature_at(now_ts, mtf_bias_map)
        if not cli_math.passes_mtf_gate(sig, net): continue

        entry_price = current_price
        stop_price = base['base_low'] * (1 - cli_cfg.STOP_OFFSET_PCT)
        risk = entry_price - stop_price
        if risk <= 0: continue
        tif_end = now_ts + pd.Timedelta(days=cli_cfg.TIF_DAYS)
        forward = f.loc[now_ts:tif_end]

        # v2 tier state machine
        state = {"t1_done":False, "t2_done":False, "trail_armed":False,
                 "high_water":entry_price, "active_stop":stop_price}
        remaining = 1.0; realized_R = 0.0; mfe_R = 0.0
        outcome_final = 'tif'
        exit_ts_final = forward.index[-1]
        exit_price_final = float(forward['spot_c'].iloc[-1])

        for ts_, bar in forward.iterrows():
            if ts_ == now_ts: continue
            bar_h=float(bar['spot_h']); bar_l=float(bar['spot_l'])
            mfe_R = max(mfe_R, (bar_h - entry_price) / risk)
            result = cli_math.evaluate_tier_transitions(
                state, bar_high=bar_h, bar_low=bar_l,
                entry=entry_price, stop_initial=stop_price,
                t1_r=cli_cfg.T1_R, t2_r=cli_cfg.T2_R, trail_pct=cli_cfg.TRAIL_PCT)
            state = result['new_state']
            for act in result['actions']:
                if act['kind'] == 't1':
                    sp = cli_cfg.T1_CLOSE_PCT
                    r = (act['price']-entry_price)/risk
                    cost = cost_per_unit*(act['price']/risk)*sp
                    realized_R += sp*r - cost; remaining -= sp
                elif act['kind'] == 't2':
                    sp = cli_cfg.T2_CLOSE_PCT * remaining
                    r = (act['price']-entry_price)/risk
                    cost = cost_per_unit*(act['price']/risk)*sp
                    realized_R += sp*r - cost; remaining -= sp
                elif act['kind'] == 'stop_exit':
                    r = (act['price']-entry_price)/risk
                    cost = cost_per_unit*(act['price']/risk)*remaining
                    realized_R += remaining*r - cost
                    outcome_final = ('trail_exit' if state.get('trail_armed') and
                                       state['active_stop'] > stop_price else 'stop')
                    exit_ts_final = ts_; exit_price_final = act['price']
                    remaining = 0.0; break
            if remaining <= 1e-9: break
        if remaining > 0:
            r = (exit_price_final-entry_price)/risk
            cost = cost_per_unit*(exit_price_final/risk)*remaining
            realized_R += remaining*r - cost
            outcome_final = 'tif'
        hold_h = (exit_ts_final-now_ts).total_seconds()/3600
        signals.append({
            'asset': asset_name, 'now_ts': now_ts,
            'entry': entry_price, 'stop': stop_price,
            'base_low': base['base_low'],
            'conf_score': cf, 'mtf_sig': sig, 'mtf_net': net,
            'outcome': outcome_final, 'r_net': realized_R,
            'mfe_R': mfe_R, 'hold_h': hold_h,
            't1_done': state['t1_done'], 't2_done': state['t2_done'],
        })
        last_trigger = now_ts
    return pd.DataFrame(signals)

results = {}
for asset in ['BTC', 'ETH', 'OP']:
    use_conf = (asset == 'BTC')
    print(f'\n=== Running backtest: {asset} (confluence={use_conf}) ===')
    df = run_backtest(asset, asset_frames[asset], asset_mtf_maps[asset],
                       use_confluence=use_conf)
    results[asset] = df
    if len(df) > 0:
        win_rate = (df['r_net']>0).mean()
        years = (df['now_ts'].max() - df['now_ts'].min()).total_seconds()/(365.25*86400)
        years = max(years, 0.1)
        print(f'  signals: {len(df)}  ({len(df)/years:.1f}/year)')
        print(f'  net mean R: {df["r_net"].mean():+.3f}')
        print(f'  win rate: {win_rate:.1%}')
        print(f'  T1 hit: {df["t1_done"].mean():.1%} | T2 hit: {df["t2_done"].mean():.1%}')
        print(f'  outcomes: {df["outcome"].value_counts().to_dict()}')

## Combined portfolio annual return

In [ ]:
all_signals = pd.concat([df for df in results.values() if len(df)>0],
                          ignore_index=True)
all_signals = all_signals.sort_values('now_ts').reset_index(drop=True)

total = len(all_signals)
span_y = (all_signals['now_ts'].max() - all_signals['now_ts'].min()).total_seconds()/(365.25*86400)
per_year = total / max(span_y, 0.1)
mean_R = all_signals['r_net'].mean() if total > 0 else 0.0
win_rate = (all_signals['r_net']>0).mean() if total > 0 else 0.0

print('=== COMBINED v3 (BTC + ETH + OP) ===')
print(f'Total signals: {total}  over {span_y:.1f}y  ({per_year:.1f}/year)')
print(f'Net mean R: {mean_R:+.3f}')
print(f'Win rate (R>0): {win_rate:.1%}')

RISK_PER_TRADE_NAV = 0.02
per_trade_ret = RISK_PER_TRADE_NAV * mean_R
ann_ret = (1 + per_trade_ret) ** per_year - 1 if per_year > 0 else 0.0
print(f'\nAt 2% risk/trade, implied annual return: {ann_ret*100:+.1f}%  (= {1+ann_ret:.2f}x)')
print()
for asset, df in results.items():
    if len(df) == 0:
        print(f'  {asset}: 0 signals — possibly indicates data issue')
        continue
    yr = (df['now_ts'].max()-df['now_ts'].min()).total_seconds()/(365.25*86400)
    per_year_a = len(df)/max(yr,0.1)
    mr = df['r_net'].mean()
    per_t = RISK_PER_TRADE_NAV * mr
    ann_a = (1+per_t)**per_year_a - 1
    print(f'  {asset}: {len(df)} signals, {per_year_a:.1f}/yr, mean R {mr:+.2f}, '
          f'implied {ann_a*100:+.0f}% annual')

## Year breakdown

In [ ]:
if total > 0:
    all_signals['year'] = pd.to_datetime(all_signals['now_ts']).dt.year
    by_y = all_signals.groupby(['asset','year']).agg(
        n=('r_net','size'), mean_R=('r_net','mean'),
        t1_rate=('t1_done','mean'),
        wr=('r_net', lambda s: (s>0).mean()),
    ).round(3)
    print(by_y)
    print()
    by_y_total = all_signals.groupby('year').agg(
        n=('r_net','size'), mean_R=('r_net','mean'),
    ).round(3)
    print('combined by year:')
    print(by_y_total)

## Verdict

Compare per-asset stats. If ETH/OP have meaningfully positive R despite
reduced filtering, the multi-asset path is real and v3.5 should backfill
missing data to lift R further. If they're around 0 or negative, the
confluence score is doing most of the work and we need the data first.